# ConcertDemandAI — EDA y entrenamiento del modelo

Este notebook forma parte del desarrollo de **ConcertDemandAI**, un sistema inteligente para estimar la demanda esperada de conciertos.

El objetivo del notebook es realizar el análisis exploratorio del dataset, validar la variable objetivo `demand_class`, incorporar la variable geográfica `country`, preparar las variables del modelo, entrenar clasificadores y evaluar el desempeño para predecir la demanda de un concierto en tres categorías: **baja**, **media** y **alta**.


## 1. Configuración del entorno

Esta sección prepara el entorno de trabajo. Si el notebook se ejecuta en Google Colab, clona el repositorio y entra a la carpeta del proyecto. Si ya estás trabajando dentro del repositorio, puedes ejecutar la celda sin problema.

## 1.1 Actualización del repositorio en Colab

Si estás ejecutando el notebook en Google Colab, esta celda permite trabajar con la versión más reciente del repositorio. Si ya tienes el repositorio clonado y actualizado, puedes omitirla.


In [ ]:
# Ejecuta esta celda solo si necesitas clonar de nuevo el repositorio en Colab.
# Si ya estás dentro de /content/concert-demand-ml, puedes omitirla.

from pathlib import Path

if Path('/content').exists():
    %cd /content
    !rm -rf concert-demand-ml
    !git clone https://github.com/leonciochiunti/concert-demand-ml.git
    %cd concert-demand-ml
else:
    print('Entorno local detectado. No se clona el repositorio.')


In [ ]:
# Verificación rápida de archivos disponibles
from pathlib import Path
import os

print('Carpeta actual:', Path.cwd())
print('Archivos/carpetas disponibles:')
print(os.listdir())


In [ ]:
# Si ya clonaste el repositorio y solo quieres actualizarlo, ejecuta esta celda.
# Si acabas de clonar con la celda anterior, puedes omitirla.

if Path('/content/concert-demand-ml').exists():
    %cd /content/concert-demand-ml
    !git pull origin main
else:
    print('No existe /content/concert-demand-ml. Primero clona el repositorio.')


##START IMPORT LIBRARIES

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/leonciochiunti/concert-demand-ml.git'
REPO_DIR = Path('/content/concert-demand-ml')

# Configuración pensada para Google Colab
if Path('/content').exists():
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    %cd /content/concert-demand-ml
else:
    print('Entorno local detectado')
    print('Carpeta actual:', Path.cwd())

print('Carpeta de trabajo:', Path.cwd())
print('Archivos/carpetas disponibles:')
print(os.listdir())


## 2. Importación de librerías

Se importan las librerías necesarias para el análisis de datos, visualización, preprocesamiento, entrenamiento, evaluación y guardado del modelo.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", None)

## 3. Carga del dataset

En esta sección se carga el archivo `data/dataset.csv`, el cual contiene registros de conciertos con variables como artista, género, ciudad, tipo de recinto, capacidad, precio del boleto, presupuesto de marketing y popularidad del artista.

In [ ]:
DATA_PATH = Path('data/dataset.csv')
MODEL_PATH = Path('model/model.pkl')
BACKUP_PATH = Path('data/dataset_backup_before_rebuild.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        'No se encontró data/dataset.csv. Verifica que estés dentro del repositorio concert-demand-ml.'
    )

df = pd.read_csv(DATA_PATH)

# Normalización ligera de nombres de ciudad para mantener consistencia en el dataset.
city_corrections = {
    'LosAngeles': 'Los Angeles',
    'MexicoCity': 'CDMX',
    'CiudadDeMexico': 'CDMX',
}

df['city'] = df['city'].replace(city_corrections)

# Si el dataset aún no tiene country, se crea a partir de city.
city_country_map = {
    'CDMX': 'Mexico',
    'Guadalajara': 'Mexico',
    'Monterrey': 'Mexico',
    'Queretaro': 'Mexico',
    'Leon': 'Mexico',
    'Puebla': 'Mexico',
    'Toluca': 'Mexico',
    'Los Angeles': 'Estados Unidos',
    'Bogota': 'Colombia',
    'Madrid': 'España',
    'Seul': 'Corea del Sur',
}

if 'country' not in df.columns:
    df['country'] = df['city'].map(city_country_map)
else:
    # Si ya existe, solo completamos valores faltantes.
    df['country'] = df['country'].fillna(df['city'].map(city_country_map))

missing_country = df[df['country'].isna()]['city'].unique()
if len(missing_country) > 0:
    print('Ciudades sin país asignado:')
    print(missing_country)
else:
    print('Todas las ciudades tienen país asignado.')

# Reordenar columnas para dejar country después de city.
columns_order = [
    'artist',
    'genre',
    'city',
    'country',
    'venue_type',
    'capacity',
    'month',
    'event_day',
    'days_until_event',
    'marketing_budget',
    'ticket_price',
    'artist_popularity',
    'occupancy_pct',
    'tickets_sold',
    'demand_class',
]

df = df[columns_order]

# Guardar el dataset actualizado para que el resto del flujo use la misma estructura.
df.to_csv(DATA_PATH, index=False)

print('Dataset cargado correctamente')
print('Filas y columnas:', df.shape)
print('Columnas:', df.columns.tolist())
display(df.head())


## 4. Análisis exploratorio de datos — EDA

El análisis exploratorio permite comprender la estructura del dataset antes de entrenar el modelo. En esta etapa se revisan columnas, tipos de datos, valores nulos, duplicados, distribución de la variable objetivo y relaciones entre variables.

También se analiza la variable `country`, agregada para distinguir el mercado geográfico de cada concierto. Esta variable puede ser útil porque la demanda puede variar según ciudad, país, tamaño del mercado y alcance del artista.


### 4.1 Estructura general del dataset

In [ ]:
print("Columnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

print("\nPrimeras filas:")
display(df.head())

### 4.2 Valores nulos y duplicados

Esta revisión permite confirmar si el dataset requiere limpieza adicional antes del modelado.

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())

print("\nRegistros duplicados:")
print(df.duplicated().sum())

### 4.3 Validación y reconstrucción de la variable objetivo

La variable objetivo `demand_class` debe tener tres clases: `baja`, `media` y `alta`. Además, debe ser coherente con el comportamiento de ocupación: baja demanda debe tener menor ocupación promedio que media, y media menor que alta.

La siguiente celda revisa si la variable objetivo está bien construida. Si detecta que está desbalanceada o incoherente, reconstruye las etiquetas con base en variables conocidas antes del concierto y guarda el dataset corregido con el mismo nombre `data/dataset.csv`.

In [ ]:
EXPECTED_CLASSES = {'baja', 'media', 'alta'}


def normalize(series):
    """Normaliza una serie entre 0 y 1."""
    denominator = series.max() - series.min()
    if denominator == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / denominator


def labels_are_valid(dataframe):
    """Valida si demand_class está balanceada y es coherente con occupancy_pct."""
    if 'demand_class' not in dataframe.columns:
        return False

    counts = dataframe['demand_class'].value_counts()
    if set(counts.index) != EXPECTED_CLASSES:
        return False

    # Validación mínima de balance. Cada clase debe tener al menos 20% del total.
    if counts.min() < len(dataframe) * 0.20:
        return False

    if 'occupancy_pct' in dataframe.columns:
        means = dataframe.groupby('demand_class')['occupancy_pct'].mean()
        if not (means['baja'] < means['media'] < means['alta']):
            return False

    return True


def rebuild_demand_labels(dataframe):
    """Reconstruye demand_class con una lógica reproducible basada en variables previas al evento."""
    data = dataframe.copy()

    # Pesos por género musical
    genre_weights = {
        'reggaeton': 1.00,
        'pop': 0.95,
        'kpop': 0.95,
        'regional_mexicano': 0.90,
        'banda': 0.85,
        'norteno': 0.85,
        'corridos': 0.85,
        'rock': 0.80,
        'electronica': 0.78,
        'rap': 0.75,
        'hiphop': 0.75,
        'indie': 0.65,
        'jazz': 0.55,
        'clasica': 0.50,
    }

    # Pesos por tipo de recinto
    venue_weights = {
        'stadium': 1.00,
        'arena': 0.90,
        'festival': 0.85,
        'theater': 0.65,
        'club': 0.50,
    }

    # Pesos por día del evento
    day_weights = {
        'Friday': 1.00,
        'Saturday': 1.00,
        'Sunday': 0.80,
        'Thursday': 0.70,
        'Wednesday': 0.60,
        'Tuesday': 0.55,
        'Monday': 0.50,
    }

    # Pesos por ciudad y país
    city_weights = {
        'CDMX': 1.00,
        'Guadalajara': 0.90,
        'Monterrey': 0.90,
        'Los Angeles': 0.88,
        'Bogota': 0.85,
        'Madrid': 0.85,
        'Seul': 0.85,
        'Queretaro': 0.75,
        'Puebla': 0.70,
        'Leon': 0.70,
        'Toluca': 0.65,
    }

    country_weights = {
        'Mexico': 0.90,
        'Estados Unidos': 0.88,
        'Colombia': 0.84,
        'España': 0.84,
        'Corea del Sur': 0.86,
    }

    popularity_score = normalize(data['artist_popularity'])
    marketing_score = normalize(data['marketing_budget'])
    capacity_score = normalize(data['capacity'])
    price_score = normalize(data['ticket_price'])
    days_score = 1 - normalize(data['days_until_event'])

    genre_score = data['genre'].map(genre_weights).fillna(0.70)
    venue_score = data['venue_type'].map(venue_weights).fillna(0.65)
    day_score = data['event_day'].map(day_weights).fillna(0.65)
    city_score = data['city'].map(city_weights).fillna(0.70)
    country_score = data['country'].map(country_weights).fillna(0.80) if 'country' in data.columns else 0.80

    np.random.seed(42)
    noise = np.random.normal(0, 0.03, size=len(data))

    data['demand_score'] = (
        0.28 * popularity_score +
        0.20 * marketing_score +
        0.15 * capacity_score +
        0.10 * genre_score +
        0.10 * venue_score +
        0.07 * city_score +
        0.05 * country_score +
        0.05 * day_score +
        0.05 * days_score -
        0.05 * price_score +
        noise
    )

    data['demand_class'] = pd.qcut(
        data['demand_score'],
        q=3,
        labels=['baja', 'media', 'alta']
    )

    def generate_occupancy(row):
        if row['demand_class'] == 'baja':
            return np.random.uniform(0.25, 0.55)
        elif row['demand_class'] == 'media':
            return np.random.uniform(0.56, 0.78)
        else:
            return np.random.uniform(0.79, 0.97)

    np.random.seed(42)
    data['occupancy_pct'] = data.apply(generate_occupancy, axis=1).round(2)
    data['tickets_sold'] = (data['capacity'] * data['occupancy_pct']).round().astype(int)

    data = data.drop(columns=['demand_score'], errors='ignore')
    return data


print('Distribución inicial de demand_class:')
print(df['demand_class'].value_counts())

if labels_are_valid(df):
    print('\nLa variable demand_class ya es válida y coherente. No se reconstruye.')
else:
    print('\nLa variable demand_class no es válida o está desbalanceada. Se reconstruirá de forma reproducible.')
    df.to_csv(BACKUP_PATH, index=False)
    print('Respaldo creado en:', BACKUP_PATH)

    df = rebuild_demand_labels(df)
    df.to_csv(DATA_PATH, index=False)
    print('Dataset actualizado y guardado en:', DATA_PATH)

print('\nDistribución final de demand_class:')
print(df['demand_class'].value_counts())

print('\nPromedio de occupancy_pct por clase:')
print(df.groupby('demand_class')['occupancy_pct'].mean().reindex(['baja', 'media', 'alta']))


### 4.4 Distribución de la variable objetivo

Se visualiza la distribución de las clases `baja`, `media` y `alta`. Un dataset balanceado ayuda a evitar que el modelo favorezca una clase dominante.

In [ ]:
class_counts = df["demand_class"].value_counts().reindex(["baja", "media", "alta"])

plt.figure(figsize=(7, 4))
class_counts.plot(kind="bar")
plt.title("Distribución de clases de demanda")
plt.xlabel("Clase de demanda")
plt.ylabel("Cantidad de registros")
plt.xticks(rotation=0)
plt.show()

print("Distribución porcentual:")
print((class_counts / len(df) * 100).round(2))

### 4.5 Análisis de variables numéricas

Se revisan medidas estadísticas de las variables numéricas para comprender sus rangos, promedios y posibles valores extremos.

In [ ]:
numeric_columns = [
    "capacity",
    "month",
    "days_until_event",
    "marketing_budget",
    "ticket_price",
    "artist_popularity",
    "occupancy_pct",
    "tickets_sold",
]

display(df[numeric_columns].describe().T)

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(7, 4))
    df[col].hist(bins=20)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

### 4.6 Análisis de variables categóricas

Se revisan variables como artista, género, ciudad, país, tipo de recinto y día del evento. Estas variables son relevantes porque representan características contextuales del concierto que pueden influir en el nivel de demanda esperado.


In [ ]:
categorical_columns = [
    "artist",
    "genre",
    "city",
    "country",
    "venue_type",
    "event_day",
    "demand_class",
]

for col in categorical_columns:
    print(f"\nVariable: {col}")
    print(df[col].value_counts().head(15))

In [ ]:
plt.figure(figsize=(10, 5))
df['genre'].value_counts().plot(kind='bar')
plt.title('Cantidad de conciertos por género musical')
plt.xlabel('Género')
plt.ylabel('Cantidad de registros')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
df['city'].value_counts().plot(kind='bar')
plt.title('Cantidad de conciertos por ciudad')
plt.xlabel('Ciudad')
plt.ylabel('Cantidad de registros')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
df['country'].value_counts().plot(kind='bar')
plt.title('Cantidad de conciertos por país')
plt.xlabel('País')
plt.ylabel('Cantidad de registros')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### 4.7 Mapa de calor de correlación

El mapa de calor permite observar relaciones entre variables numéricas. Las variables `occupancy_pct` y `tickets_sold` se incluyen únicamente para análisis, pero se excluyen del entrenamiento porque representan resultados posteriores al evento.

In [ ]:
corr = df[numeric_columns].corr()

plt.figure(figsize=(10, 6))
plt.imshow(corr, cmap="coolwarm", interpolation="nearest")
plt.colorbar(label="Correlación")

plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            color="black",
            fontsize=8,
        )

plt.title("Mapa de calor de correlación entre variables numéricas")
plt.tight_layout()
plt.show()

### 4.8 Análisis por clase de demanda

Se comparan variables relevantes contra `demand_class` para revisar si las clases tienen comportamiento lógico.

In [ ]:
occupancy_by_class = df.groupby("demand_class")["occupancy_pct"].mean().reindex(["baja", "media", "alta"])

plt.figure(figsize=(7, 4))
occupancy_by_class.plot(kind="bar")
plt.title("Promedio de ocupación por clase de demanda")
plt.xlabel("Clase de demanda")
plt.ylabel("Promedio de ocupación")
plt.xticks(rotation=0)
plt.show()

print("Promedio de ocupación por clase:")
print(occupancy_by_class)

In [ ]:
pd.crosstab(df['venue_type'], df['demand_class'])[['baja', 'media', 'alta']].plot(
    kind='bar',
    figsize=(10, 5)
)
plt.title('Distribución de demanda por tipo de recinto')
plt.xlabel('Tipo de recinto')
plt.ylabel('Cantidad de conciertos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Clase de demanda')
plt.tight_layout()
plt.show()

pd.crosstab(df['event_day'], df['demand_class'])[['baja', 'media', 'alta']].plot(
    kind='bar',
    figsize=(10, 5)
)
plt.title('Distribución de demanda por día del evento')
plt.xlabel('Día del evento')
plt.ylabel('Cantidad de conciertos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Clase de demanda')
plt.tight_layout()
plt.show()

pd.crosstab(df['country'], df['demand_class'])[['baja', 'media', 'alta']].plot(
    kind='bar',
    figsize=(8, 5)
)
plt.title('Distribución de demanda por país')
plt.xlabel('País')
plt.ylabel('Cantidad de conciertos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Clase de demanda')
plt.tight_layout()
plt.show()


### 4.9 Revisión de posibles valores atípicos

Se revisan posibles valores extremos en variables numéricas. Estos valores no se eliminan automáticamente, porque pueden representar eventos premium, conciertos masivos o diferencias reales de escala.

In [ ]:
for col in ["capacity", "marketing_budget", "ticket_price", "artist_popularity"]:
    plt.figure(figsize=(7, 4))
    df.boxplot(column=col, by="demand_class")
    plt.title(f"Distribución de {col} por clase de demanda")
    plt.suptitle("")
    plt.xlabel("Clase de demanda")
    plt.ylabel(col)
    plt.show()

In [ ]:
def iqr_outlier_summary(dataframe, columns):
    rows = []
    for col in columns:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = dataframe[(dataframe[col] < lower) | (dataframe[col] > upper)]
        rows.append({
            "variable": col,
            "limite_inferior": lower,
            "limite_superior": upper,
            "outliers": len(outliers),
        })
    return pd.DataFrame(rows)

outlier_summary = iqr_outlier_summary(
    df,
    ["capacity", "marketing_budget", "ticket_price", "artist_popularity"]
)

display(outlier_summary)

## 5. Preparación de variables para el modelo

Para entrenar el modelo se seleccionan únicamente variables disponibles antes del concierto. Se excluyen `occupancy_pct` y `tickets_sold` para evitar fuga de información.

In [ ]:
features = [
    'artist',
    'genre',
    'city',
    'country',
    'venue_type',
    'capacity',
    'month',
    'event_day',
    'days_until_event',
    'marketing_budget',
    'ticket_price',
    'artist_popularity',
]

target = 'demand_class'

leakage_columns = ['occupancy_pct', 'tickets_sold']

for col in leakage_columns:
    assert col not in features, f'La columna {col} no debe usarse como entrada del modelo.'

X = df[features]
y = df[target]

categorical_features = [
    'artist',
    'genre',
    'city',
    'country',
    'venue_type',
    'event_day',
]

numeric_features = [
    'capacity',
    'month',
    'days_until_event',
    'marketing_budget',
    'ticket_price',
    'artist_popularity',
]

print('Variables de entrada:')
print(features)

print('\nVariable objetivo:')
print(target)

print('\nVariables categóricas:')
print(categorical_features)

print('\nVariables numéricas:')
print(numeric_features)

print('\nTamaño de X:', X.shape)
print('Tamaño de y:', y.shape)


## 6. División de datos

Se divide el dataset en entrenamiento y prueba usando una proporción 80/20. La división es estratificada para conservar el balance de clases en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:", X_test.shape)

print("\nDistribución en entrenamiento:")
print(y_train.value_counts())

print("\nDistribución en prueba:")
print(y_test.value_counts())

## 7. Preprocesamiento y entrenamiento

Se utiliza `OneHotEncoder` para variables categóricas y se dejan pasar las variables numéricas. Después se comparan varios modelos para seleccionar una opción adecuada.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

models = {
    "Dummy_baseline": DummyClassifier(strategy="most_frequent"),
    "DecisionTree_depth6": DecisionTreeClassifier(
        max_depth=6,
        random_state=42,
        class_weight="balanced",
    ),
    "RandomForest_depth8": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42,
        class_weight="balanced",
    ),
    "RandomForest_depth12": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        class_weight="balanced",
    ),
}

trained_pipelines = {}
results = []

for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)
    gap = train_accuracy - test_accuracy

    trained_pipelines[name] = pipeline
    results.append({
        "modelo": name,
        "accuracy_train": train_accuracy,
        "accuracy_test": test_accuracy,
        "gap_train_test": gap,
    })

results_df = pd.DataFrame(results).sort_values("accuracy_test", ascending=False)
display(results_df)

### 7.1 Selección del modelo final

Se selecciona el modelo con mejor equilibrio entre accuracy de prueba y diferencia razonable entre entrenamiento y prueba. Esto ayuda a evitar elegir un modelo que memorice demasiado el conjunto de entrenamiento.

In [ ]:
# Se priorizan modelos con una brecha train-test no mayor a 0.12.
reasonable_candidates = results_df[results_df["gap_train_test"] <= 0.12]

if len(reasonable_candidates) > 0:
    selected_model_name = reasonable_candidates.sort_values("accuracy_test", ascending=False).iloc[0]["modelo"]
else:
    selected_model_name = results_df.iloc[0]["modelo"]

best_pipeline = trained_pipelines[selected_model_name]

print("Modelo seleccionado:", selected_model_name)
print("\nResultados del modelo seleccionado:")
display(results_df[results_df["modelo"] == selected_model_name])

## 8. Evaluación del modelo

Se evalúa el modelo final mediante accuracy, reporte de clasificación y matriz de confusión.

In [ ]:
y_pred = best_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy en prueba:", round(accuracy, 4))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, labels=["baja", "media", "alta"], zero_division=0))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred, labels=["baja", "media", "alta"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["baja", "media", "alta"],
)
plt.title(f"Matriz de confusión - {selected_model_name}")
plt.show()

### 8.1 Importancia de variables

Para interpretar el modelo seleccionado, se revisan las variables con mayor importancia. Esto ayuda a entender qué factores aportan más a la predicción de demanda.

In [ ]:
model_step = best_pipeline.named_steps["model"]
preprocessor_step = best_pipeline.named_steps["preprocessor"]

if hasattr(model_step, "feature_importances_"):
    transformed_feature_names = preprocessor_step.get_feature_names_out()
    importances = model_step.feature_importances_

    importance_df = pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    display(importance_df.head(20))

    plt.figure(figsize=(10, 6))
    importance_df.head(15).sort_values("importance").plot(
        x="feature",
        y="importance",
        kind="barh",
        legend=False,
        figsize=(10, 6),
    )
    plt.title("Top 15 variables más importantes")
    plt.xlabel("Importancia")
    plt.ylabel("Variable")
    plt.tight_layout()
    plt.show()
else:
    print("El modelo seleccionado no expone feature_importances_.")

## 9. Guardado del modelo

El modelo entrenado se guarda en `model/model.pkl`, para que pueda cargarse después desde `model/predict.py` o desde la aplicación Streamlit.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, MODEL_PATH)

print("Modelo guardado correctamente en:", MODEL_PATH)

## 10. Prueba de predicción manual

Se realiza una prueba con un concierto ficticio para validar que el modelo pueda recibir datos nuevos y devolver una clase de demanda junto con las probabilidades por clase.

In [ ]:
example = {
    'artist': 'Peso Pluma',
    'genre': 'regional_mexicano',
    'city': 'CDMX',
    'country': 'Mexico',
    'venue_type': 'stadium',
    'capacity': 45000,
    'month': 11,
    'event_day': 'Friday',
    'days_until_event': 60,
    'marketing_budget': 500000,
    'ticket_price': 1800,
    'artist_popularity': 94,
}

input_df = pd.DataFrame([example])

prediction = best_pipeline.predict(input_df)[0]
probabilities = best_pipeline.predict_proba(input_df)[0]

print('Datos de entrada:')
display(input_df)

print('Predicción:', prediction)

print('\nProbabilidades por clase:')
for clase, prob in zip(best_pipeline.classes_, probabilities):
    print(f'{clase}: {prob:.4f}')


## 11. Conclusiones

A partir del análisis exploratorio realizado, se concluye que el dataset utilizado para **ConcertDemandAI** tiene una estructura adecuada para construir un modelo inicial de clasificación de demanda de conciertos. El conjunto de datos contiene registros con variables relevantes para el contexto del evento, como artista, género musical, ciudad, país, tipo de recinto, capacidad, mes, día del evento, días restantes, presupuesto de marketing, precio del boleto y popularidad del artista.

La incorporación de la variable `country` permite distinguir el mercado geográfico al que pertenece cada concierto. Esto refuerza el análisis porque la demanda de un evento no depende únicamente del artista o el recinto, sino también del contexto del país y la ciudad donde se realiza.

Durante la revisión inicial se validó la presencia de valores nulos, duplicados y la distribución de la variable objetivo `demand_class`. La variable objetivo quedó organizada en tres clases: **baja**, **media** y **alta**, con una distribución balanceada. Esto favorece el entrenamiento del modelo, ya que reduce el riesgo de sesgo hacia una clase dominante.

El análisis de `occupancy_pct` mostró un comportamiento coherente con la clasificación de demanda: los conciertos de demanda baja presentan menor ocupación promedio, los de demanda media presentan valores intermedios y los de demanda alta presentan los mayores niveles de ocupación. Esto permite validar que la clasificación construida tiene sentido respecto al comportamiento esperado de los eventos.

En el análisis de correlación se incluyeron variables como capacidad, presupuesto de marketing, precio del boleto, popularidad del artista, porcentaje de ocupación y boletos vendidos. Sin embargo, `occupancy_pct` y `tickets_sold` no se usaron como variables predictoras, ya que representan información posterior al evento y su inclusión produciría fuga de información.

Para el modelado se utilizaron únicamente variables conocidas antes del concierto, incluyendo `country` como variable categórica. Se comparó un modelo base con modelos de árbol y ensamble, seleccionando el modelo con mejor equilibrio entre desempeño en prueba y control de sobreajuste. La evaluación mediante accuracy, reporte de clasificación y matriz de confusión permite analizar la capacidad del sistema para distinguir entre demanda baja, media y alta.

En conclusión, el notebook demuestra que es viable construir un primer modelo de clasificación para apoyar la estimación de demanda en conciertos. Aunque el dataset es simulado, el flujo desarrollado permite validar la lógica del MVP y sirve como base para una futura integración con datos reales provenientes de plataformas de venta de boletos, históricos de asistencia o fuentes públicas de eventos.


## 12. Trabajo futuro

Como trabajo futuro, se propone mejorar el dataset incorporando datos reales de conciertos, ventas históricas, precios por zona, ubicación geográfica, comportamiento de compra y tendencias por artista. Esto permitiría entrenar modelos con mayor capacidad de generalización en escenarios reales.

También se recomienda comparar más algoritmos de clasificación y realizar validación cruzada para obtener una evaluación más robusta. Además, se podría mejorar la interfaz de Streamlit para permitir al usuario ingresar datos de un concierto y visualizar la predicción de demanda de forma clara.

Finalmente, el proyecto puede ampliarse con un sistema de recomendación de conciertos basado en similitud de características y, posteriormente, con un módulo de procesamiento de lenguaje natural para permitir consultas en lenguaje natural.